# IA-01 — Modelo sustituto del factor Z con una red neuronal

Primer ejemplo de la línea "proponer nuevas soluciones con IA". Se entrena un perceptrón multicapa que
aprende $Z(p_{pr}, T_{pr})$ a partir de la correlación de Dranchuk-Purvis-Robinson del Cap. 1, y se compara
precisión y velocidad.

**Preguntas para clase.** ¿Dónde falla el modelo? ¿Qué pasa fuera del dominio de entrenamiento? ¿Por qué
importa que un sustituto respete la física (Z → 1 cuando p → 0)?

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from ductos.pvt_gas import z_dpr

rng = np.random.default_rng(0)
n = 4000
ppr = rng.uniform(0.2, 15, n)
tpr = rng.uniform(1.05, 3.0, n)
z = np.array([z_dpr(p, t) for p, t in zip(ppr, tpr)])
X = np.column_stack([ppr, tpr])
corte = int(0.8 * n)
modelo = make_pipeline(StandardScaler(), MLPRegressor(hidden_layer_sizes=(64, 64), max_iter=3000,
                                                      tol=1e-7, random_state=0))
modelo.fit(X[:corte], z[:corte])
err = modelo.predict(X[corte:]) - z[corte:]
print(f"Error absoluto medio en prueba: {np.mean(np.abs(err)):.4f}   máximo: {np.max(np.abs(err)):.4f}")

In [ ]:
malla_p = np.linspace(0.2, 15, 120)
fig, axs = plt.subplots(1, 2, figsize=(12, 4.5))
for t in [1.05, 1.2, 1.5, 2.0, 3.0]:
    zr = [z_dpr(p, t) for p in malla_p]
    zm = modelo.predict(np.column_stack([malla_p, np.full_like(malla_p, t)]))
    l, = axs[0].plot(malla_p, zr, label=f"Tpr={t}")
    axs[0].plot(malla_p, zm, "--", c=l.get_color())
    axs[1].plot(malla_p, zm - zr, c=l.get_color())
axs[0].set(xlabel="ppr", ylabel="Z", title="DPR (línea) vs. red neuronal (guiones)")
axs[1].set(xlabel="ppr", ylabel="error", title="Error del sustituto")
for a in axs: a.grid(alpha=0.3)
axs[0].legend(fontsize=8); plt.tight_layout(); plt.show()

In [ ]:
Xq = np.column_stack([rng.uniform(0.2, 15, 20000), rng.uniform(1.05, 3, 20000)])
t0 = time.perf_counter(); modelo.predict(Xq); t_nn = time.perf_counter() - t0
t0 = time.perf_counter(); [z_dpr(p, t) for p, t in Xq]; t_dpr = time.perf_counter() - t0
print(f"20 000 evaluaciones -> DPR iterativo: {t_dpr:.2f} s | red neuronal: {t_nn:.3f} s")
print(f"Extrapolación (ppr=0.01, Tpr=1.5): DPR {z_dpr(0.01, 1.5):.4f} | red {modelo.predict([[0.01, 1.5]])[0]:.4f}")

## Siguientes propuestas (hoja de ruta IA)

1. **Sustitutos con restricciones físicas**: imponer $Z(0,T)=1$ y monotonía en la arquitectura o en la pérdida.
2. **PINN para flujo transitorio en gasoductos** (empaque de línea, *line pack*): resolver las ecuaciones de
   continuidad y momento con la Ec. 4.19 como caso estacionario de validación.
3. **Selección de correlación asistida por LLM**: un agente que, con los datos de campo, recomiende la ecuación de
   la Tabla 4.4 y explique por qué, citando la sección del libro.
4. **Calibración bayesiana de la eficiencia E** con mediciones de campo.
5. **Optimización de redes** (Graper, Lagrange del Cap. 4.7) con optimizadores modernos y restricciones de diámetros comerciales.